# Isolated PP1 Vision exploratory scaffold v1

**Provisional labels; non-clinical; non-final; participant independence unverified.**

Implementation and synthetic verification only are authorised now. Real dental image packaging/training is NOT authorised. No Test data, Drive mount, installs or pretrained downloads. Use a fresh runtime; choose GPU for the future pilot. The synthetic checks work on CPU without pretrained weights.

Today: upload ONLY `vision_pp1_exploratory_code_v1.zip`, run code verification/environment/synthetic cells and collect their evidence. **Keep RUN_PILOT=False.** Historical Colab v1/v2 tests do not verify this new wrapper.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, os, stat, sys, zipfile

NOTICE = 'Provisional labels; non-clinical; non-final; participant independence unverified.'
print(NOTICE)
ROOT = Path('/content/pp1_exploratory_v1')
CODE_ROOT = ROOT / 'code'
CODE_ARCHIVE = 'vision_pp1_exploratory_code_v1.zip'
CODE_SHA256 = 'bdbd49b0f71e297d87ca17ff7f262964ed5996cf836821cf5f37737d290ce418'
EXPECTED_SOURCES = ['src/vision/model_pipeline.py', 'src/vision/train_pipeline.py', 'src/vision/exploratory_bundle.py', 'src/vision/exploratory_pilot.py', 'tests/test_exploratory_bundle.py', 'tests/test_exploratory_pilot.py']
if Path('/content/drive').exists() or CODE_ROOT.exists():
    raise RuntimeError('Use a fresh runtime without Drive; preserve any existing files.')
uploaded = files.upload()
if set(uploaded) != {CODE_ARCHIVE}:
    raise RuntimeError('Upload only the code-only ZIP; no dataset archives.')
if hashlib.sha256(uploaded[CODE_ARCHIVE]).hexdigest() != CODE_SHA256:
    raise RuntimeError('Code archive SHA-256 mismatch')
with zipfile.ZipFile(CODE_ARCHIVE) as archive:
    entries = archive.infolist()
    expected = set(EXPECTED_SOURCES) | {'code_manifest.json'}
    if len(entries) != len(expected) or {e.filename for e in entries} != expected:
        raise RuntimeError('Unexpected code archive contents')
    if sum(e.file_size for e in entries) > 16 * 1024**2:
        raise RuntimeError('Code archive size limit')
    for entry in entries:
        p = PurePosixPath(entry.filename)
        if p.is_absolute() or '..' in p.parts or stat.S_IFMT(entry.external_attr >> 16) not in {0, stat.S_IFREG}:
            raise RuntimeError('Unsafe code entry')
    code_manifest = json.loads(archive.read('code_manifest.json'))
    if set(code_manifest['files']) != set(EXPECTED_SOURCES) or code_manifest['dental_data_included'] is not False:
        raise RuntimeError('Code manifest mismatch')
    CODE_ROOT.mkdir(parents=True)
    for entry in entries:
        target = CODE_ROOT / entry.filename
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(entry))
for name, digest in code_manifest['files'].items():
    if hashlib.sha256((CODE_ROOT / name).read_bytes()).hexdigest() != digest:
        raise RuntimeError('Source hash mismatch: ' + name)
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print(NOTICE, '\nAll six source hashes verified. No dental data uploaded.')


In [ ]:
import importlib.util
required = ['torch', 'torchvision', 'numpy', 'PIL', 'pytest']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing dependencies; no automatic installation: ' + str(missing))
import torch, torchvision, numpy, PIL, pytest
from dataclasses import asdict
from src.vision import exploratory_bundle as eb
from src.vision import exploratory_pilot as ep

runtime = {'notice': NOTICE, 'python': sys.version, 'torch': str(torch.__version__),
           'torchvision': torchvision.__version__, 'numpy': numpy.__version__,
           'Pillow': PIL.__version__, 'pytest': pytest.__version__,
           'cuda_available': torch.cuda.is_available(),
           'gpu': torch.cuda.get_device_name() if torch.cuda.is_available() else None}
print(json.dumps(runtime, indent=2))
print(json.dumps(ep.pretrained_status(), indent=2))
print('Proposed profile only:', asdict(ep.configuration()))
print('No pretrained model has been loaded or downloaded.')


In [ ]:
import subprocess
import xml.etree.ElementTree as ET

print(NOTICE)
environment = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',
                   PYTHONDONTWRITEBYTECODE='1', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
completed = subprocess.run([sys.executable, '-B', '-m', 'pytest', '-q', '-rs', '-p', 'no:cacheprovider',
                           '--junitxml=pilot_synthetic_checks.xml',
                           'tests/test_exploratory_bundle.py', 'tests/test_exploratory_pilot.py'],
                          env=environment, capture_output=True, text=True)
log = NOTICE + '\n' + completed.stdout + completed.stderr
Path('pilot_synthetic_pytest.log').write_text(log)
print(log)
xml = ET.parse('pilot_synthetic_checks.xml').getroot()
suites = list(xml) if xml.tag == 'testsuites' else [xml]
counts = {key: sum(int(s.get(key, 0)) for s in suites) for key in ['tests', 'failures', 'errors', 'skipped']}
counts['passed'] = counts['tests'] - counts['failures'] - counts['errors'] - counts['skipped']
after_hashes = {name: eb.sha256(CODE_ROOT / name) for name in EXPECTED_SOURCES}
if after_hashes != code_manifest['files']:
    raise RuntimeError('Source files changed during verification')
verification = {'notice': NOTICE, 'mode': 'synthetic_only', 'counts': counts, 'runtime': runtime,
                'source_hashes': after_hashes, 'code_archive_sha256': CODE_SHA256,
                'dental_training_run': False, 'test_data_used': False}
Path('pilot_runtime_verification.json').write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))
if completed.returncode != 0 or counts['failures'] or counts['errors'] or counts['skipped']:
    raise RuntimeError('New scaffold verification incomplete; do not proceed to dental data.')


In [ ]:
print(NOTICE)
evidence = CODE_ROOT / 'vision_pp1_scaffold_runtime_evidence_v1.zip'
if evidence.exists():
    raise RuntimeError('Evidence ZIP already exists; preserve it.')
with zipfile.ZipFile(evidence, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['pilot_runtime_verification.json', 'pilot_synthetic_pytest.log', 'pilot_synthetic_checks.xml', 'code_manifest.json']:
        archive.write(CODE_ROOT / name, name)
files.download(str(evidence))


## STOP — separate real-data approval required

Do not continue today. The scaffold is authorised; real image packaging/training is not.

After separately approved Mac packaging, the owner must provide the exact development bundle ZIP and a `run_authorization.json` with cohort/code/config/bundle/weight hashes, a new run ID and all limitation acknowledgements. The original qualified manifest remains unchanged. If documented ResNet-50 V2 weights are absent, obtain separate permission to supply the exact official checkpoint. No automatic download.

Only after recording that approval may the owner deliberately change `RUN_PILOT` below. No Drive mount or full dataset upload. Results cannot be presented as clinical/final or participant-independent evidence.


In [ ]:
RUN_PILOT = False  # Remains OFF until separate explicit owner authorisation.
if not RUN_PILOT:
    print(NOTICE)
    print('STOP: no real development images packaged or trained by this notebook.')
else:
    if counts['failures'] or counts['errors'] or counts['skipped']:
        raise RuntimeError('Complete synthetic verification first')
    if Path('/content/drive').exists() or not torch.cuda.is_available():
        raise RuntimeError('Fresh isolated GPU runtime required; no CPU fallback')
    for name, digest in code_manifest['files'].items():
        if eb.sha256(CODE_ROOT / name) != digest:
            raise RuntimeError('Code changed after verification')
    incoming = files.upload()
    auth_name = 'run_authorization.json'
    if auth_name not in incoming:
        raise RuntimeError('Provide separately confirmed run_authorization.json')
    auth = json.loads(incoming[auth_name])
    archives = [name for name in incoming if name.endswith('.zip')]
    allowed = {auth_name, ep.WEIGHTS_NAME} | set(archives)
    if len(archives) != 1 or set(incoming) - allowed:
        raise RuntimeError('Only one approved development ZIP, authorization and optional approved weight file')
    archive_name = archives[0]
    bindings = {'source_draft_sha256': eb.DRAFT_SHA, 'archive_sha256': eb.sha256(archive_name),
                'code_sha256': eb.json_digest(eb.source_code_hashes()),
                'config_sha256': eb.json_digest(asdict(ep.configuration())),
                'weights_source_url': ep.WEIGHTS_URL, 'output_run_id': 'pp1-v1-run01'}
    eb.authorise(auth, 'run_exploratory', bindings, True)
    if ep.WEIGHTS_NAME in incoming:
        digest = eb.sha256(ep.WEIGHTS_NAME)
        if digest != auth.get('weights_sha256') or not digest.startswith('11ad3fa6'):
            raise RuntimeError('Supplied weight provenance mismatch')
        cache = Path(torch.hub.get_dir()) / 'checkpoints'
        cache.mkdir(parents=True, exist_ok=True)
        target = cache / ep.WEIGHTS_NAME
        if target.exists():
            if eb.sha256(target) != digest:
                raise RuntimeError('Existing cached weights differ; do not overwrite')
        else:
            import shutil
            shutil.copyfile(ep.WEIGHTS_NAME, target)
    weights = ep.pretrained_status()
    if weights['status'] != 'documented_cache_present' or weights['sha256'] != auth.get('weights_sha256'):
        raise RuntimeError('Approved cached weights missing; no download')
    data_root = ROOT / 'data/pilot_v1'
    eb.extract_bundle(archive_name, data_root, auth, True)
    output = CODE_ROOT / 'models/exploratory_pp1/pp1-v1-run01'
    result = ep.run_pilot(data_root, auth, output, True)
    print(NOTICE)
    print(json.dumps(result, indent=2, allow_nan=False))
